# Fruit Classification
### Multi-Class Deep Learning Benchmark on 13 Fruit Categories
This study benchmarks six computer vision architectures on a 13-class agricultural image dataset:
- Custom Convolutional Neural Network
- MobileNetV3-Large
- YOLOv8 Classifier
- YOLO26 Classifier
- EfficientNet-B0
- ResNet-50

**Target Fruit Categories:**
`banana`, `dragonfruit`, `jackfruit`, `lemon`, `mango`, `pineapple`, `star_fruit`, `custard_apple`, `guava`, `jujube`, `lychee`, `papaya`, `sapodilla`

**Pipeline Outline:**
1. Environment Setup and Reproducibility
2. Dataset Discovery and Exploratory Data Analysis
3. Stratified Splitting (75% Train, 15% Validation, 10% Test)
4. Data Augmentation and Loaders
5. Training and Evaluation Pipeline
6. Architectural Evaluation (Models 1 through 6)
7. Final Performance Comparison


## 1. Environment Setup and Reproducibility
Configuration of runtime dependencies, random seeds, hardware acceleration, and output directories.


In [ ]:
# Dependencies installation
!pip install -q ultralytics scikit-learn seaborn matplotlib pandas torchvision torch optuna tqdm

import os
import sys
import glob
import math
import random
import shutil
import time
import gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, f1_score
from sklearn.preprocessing import label_binarize

from ultralytics import YOLO

# Plot styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

# Execution environment detection
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle') or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
ENV_NAME = "Google Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "Local Machine")

print(f"Environment: {ENV_NAME}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda:0")
    torch.backends.cudnn.benchmark = True
else:
    print("Warning: CUDA is unavailable. Running on CPU.")
    device = torch.device("cpu")


In [ ]:
# Seed configuration for reproducibility
SEED = 42

def seed_everything(seed=42):
    """Seed Python, NumPy, and PyTorch for deterministic execution."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)
print(f"Random seed locked: {SEED}")


In [ ]:
# Output directory initialization
OUTPUT_DIR = Path("outputs")
MODELS_DIR = OUTPUT_DIR / "models"
PLOTS_DIR = OUTPUT_DIR / "plots"
REPORTS_DIR = OUTPUT_DIR / "reports"

for p in [OUTPUT_DIR, MODELS_DIR, PLOTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)


### Dataset Download & Extract


In [ ]:
# !python -m pip install --upgrade gdown -q
# !gdown "https://drive.google.com/file/d/1RNFxIXg9lvxGuI-LwA6vh_ZsMnwrB_i8/view?usp=drive_link" -q
# !mkdir -p original_dataset && cd "original_dataset" && tar -xf /content/fruit_dataset.tar


## 2. Dataset Discovery and Exploratory Data Analysis
Cataloging image counts per fruit category and inspecting quality distributions.


In [ ]:
EXPECTED_FRUITS = [
    'banana', 'dragonfruit', 'jackfruit', 'lemon', 'mango',
    'pineapple', 'star_fruit', 'custard_apple', 'guava',
    'jujube', 'lychee', 'papaya', 'sapodilla'
]

EXPECTED_QUALITIES = ['good', 'medium', 'bad']
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def locate_dataset_dir():
    """Locate the dataset directory across execution environments."""
    candidates = [
        Path("original_dataset"),
        Path("/content/original_dataset"),
        Path("/kaggle/working/original_dataset")
    ]
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob("original_dataset"):
            if p.is_dir():
                candidates.insert(0, p)
        for p in Path("/kaggle/input").glob("*"):
            if p.is_dir() and any((p / fruit).exists() for fruit in EXPECTED_FRUITS):
                candidates.insert(0, p)

    for c in candidates:
        if c.exists() and any((c / fruit).exists() for fruit in EXPECTED_FRUITS):
            return c
    return Path("original_dataset")

ORIGINAL_DATASET_DIR = locate_dataset_dir()

def scan_dataset(base_dir: Path):
    if not base_dir.exists():
        raise FileNotFoundError(f"Directory '{base_dir}' not found.")

    records = []
    for fruit in EXPECTED_FRUITS:
        fruit_dir = base_dir / fruit
        if not fruit_dir.exists():
            continue

        for quality in EXPECTED_QUALITIES:
            quality_dir = fruit_dir / quality
            if quality_dir.exists():
                images = [f for f in quality_dir.iterdir() if f.suffix.lower() in VALID_EXTENSIONS and f.is_file()]
                for img_path in images:
                    records.append({
                        "fruit": fruit,
                        "quality": quality,
                        "file_name": img_path.name,
                        "path": str(img_path)
                    })

        direct_images = [f for f in fruit_dir.iterdir() if f.suffix.lower() in VALID_EXTENSIONS and f.is_file()]
        for img_path in direct_images:
            records.append({
                "fruit": fruit,
                "quality": "unspecified",
                "file_name": img_path.name,
                "path": str(img_path)
            })

    return pd.DataFrame(records)

df_summary = scan_dataset(ORIGINAL_DATASET_DIR)
print(f"Total dataset samples: {len(df_summary)}")
df_summary.head()


In [ ]:
# Class distribution visualization
if not df_summary.empty:
    pivot_table = pd.crosstab(df_summary['fruit'], df_summary['quality'], margins=True, margins_name="Total")
    display(pivot_table)
    pivot_table.to_csv(REPORTS_DIR / "dataset_distribution_summary.csv")

    plt.figure(figsize=(12, 5))
    order = df_summary['fruit'].value_counts().index
    sns.countplot(data=df_summary, x='fruit', order=order, hue='quality', palette='Set2')
    plt.xticks(rotation=45, ha='right')
    plt.title("Sample Distribution per Fruit Category", fontsize=14)
    plt.xlabel("Category")
    plt.ylabel("Sample Count")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "eda_fruit_quality_distribution.png", dpi=300)
    plt.show()


## 3. Stratified Partitioning (75% Train, 15% Val, 10% Test)
Inner quality subfolders are merged per fruit. Stratification preserves category representation across all partitions.


In [ ]:
TARGET_DATA_DIR = Path("data")

def prepare_stratified_split(df: pd.DataFrame, target_base: Path, train_ratio=0.75, val_ratio=0.15, test_ratio=0.10):
    assert math.isclose(train_ratio + val_ratio + test_ratio, 1.0), "Split ratios must sum to 1.0"

    if target_base.exists():
        shutil.rmtree(target_base)

    for split in ['train', 'val', 'test']:
        for fruit in EXPECTED_FRUITS:
            (target_base / split / fruit).mkdir(parents=True, exist_ok=True)

    temp_val_ratio = val_ratio / (train_ratio + val_ratio)

    train_val_df, test_df = train_test_split(
        df,
        test_size=test_ratio,
        stratify=df['fruit'],
        random_state=SEED
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=temp_val_ratio,
        stratify=train_val_df['fruit'],
        random_state=SEED
    )

    splits = {'train': train_df, 'val': val_df, 'test': test_df}

    print(f"Dataset partitions:")
    print(f"  Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
    print(f"  Val:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
    print(f"  Test:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

    copy_records = []
    for split_name, split_data in splits.items():
        pbar = tqdm(split_data.iterrows(), total=len(split_data), desc=f"Preparing {split_name} set", leave=False)
        for idx, row in pbar:
            src_path = Path(row['path'])
            fruit = row['fruit']
            quality = row['quality']
            dst_filename = f"{quality}_{src_path.name}"
            dst_path = target_base / split_name / fruit / dst_filename

            shutil.copy2(src_path, dst_path)
            copy_records.append({
                "split": split_name,
                "fruit": fruit,
                "file": dst_filename,
                "path": str(dst_path)
            })

    df_splits = pd.DataFrame(copy_records)
    return df_splits

df_splits = prepare_stratified_split(df_summary, TARGET_DATA_DIR)


In [ ]:
# Partition verification table
split_summary = pd.crosstab(df_splits['fruit'], df_splits['split'])
split_summary['Total'] = split_summary.sum(axis=1)
split_summary['Train%'] = (split_summary['train'] / split_summary['Total'] * 100).round(1)
split_summary['Val%'] = (split_summary['val'] / split_summary['Total'] * 100).round(1)
split_summary['Test%'] = (split_summary['test'] / split_summary['Total'] * 100).round(1)
display(split_summary)
split_summary.to_csv(REPORTS_DIR / "stratified_split_verification.csv")


## 4. Data Augmentation and Loaders
Geometric and photometric augmentations applied to training data to improve generalization.
Standard ImageNet statistics are used for input normalization.


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2

# ImageNet normalization statistics
NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# Training pipeline with data augmentation
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    NORMALIZE
])

# Deterministic validation and test transforms
val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    NORMALIZE
])

train_dataset = datasets.ImageFolder(root="data/train", transform=train_transforms)
val_dataset = datasets.ImageFolder(root="data/val", transform=val_test_transforms)
test_dataset = datasets.ImageFolder(root="data/test", transform=val_test_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

class_names = train_dataset.classes
NUM_CLASSES = len(class_names)

print(f"Classes ({NUM_CLASSES}): {class_names}")
print(f"Batch configuration: {BATCH_SIZE} images per batch (Train: {len(train_loader)}, Val: {len(val_loader)})")


## 5. Training and Evaluation Infrastructure
Standardized training loops, metric extraction, and evaluation plotting functions.


In [ ]:
all_model_results = {}

def train_pytorch_pipeline(model, train_loader, val_loader, criterion, optimizer, scheduler=None, num_epochs=15, patience=4, model_name="model"):
    """Standardized PyTorch training engine with validation tracking, Macro F1 checkpointing, and early stopping."""
    slug = model_name.lower().replace(" ", "_").replace("-", "_")
    checkpoint_path = MODELS_DIR / f"best_{slug}.pth"

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'train_f1': [], 'val_f1': []
    }
    best_val_f1 = 0.0
    early_stop_counter = 0
    start_time = time.time()

    use_amp = (device.type == 'cuda')
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    for epoch in range(num_epochs):
        epoch_start = time.time()

        # Training
        model.train()
        running_loss, running_corrects, total_train = 0.0, 0, 0
        train_preds, train_targets = [], []
        train_pbar = tqdm(train_loader, desc=f"{model_name} [Epoch {epoch+1:02d}/{num_epochs:02d}] Train", leave=False)

        for inputs, labels in train_pbar:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total_train += labels.size(0)

            train_preds.extend(preds.detach().cpu().numpy())
            train_targets.extend(labels.detach().cpu().numpy())

            train_pbar.set_postfix({
                'loss': f"{running_loss/total_train:.4f}",
                'acc': f"{(running_corrects.double()/total_train).item()*100:.2f}%"
            })

        if scheduler:
            scheduler.step()

        train_loss = running_loss / total_train if total_train > 0 else 0
        train_acc = (running_corrects.double() / total_train).item() if total_train > 0 else 0
        train_f1 = f1_score(train_targets, train_preds, average='macro', zero_division=0)

        # Validation
        model.eval()
        val_running_loss, val_running_corrects, total_val = 0.0, 0, 0
        val_preds, val_targets = [], []
        val_pbar = tqdm(val_loader, desc=f"{model_name} [Epoch {epoch+1:02d}/{num_epochs:02d}] Val", leave=False)

        with torch.no_grad():
            for inputs, labels in val_pbar:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)
                val_running_loss += loss.item() * inputs.size(0)
                val_running_corrects += torch.sum(preds == labels.data)
                total_val += labels.size(0)

                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(labels.cpu().numpy())

                val_pbar.set_postfix({
                    'val_loss': f"{val_running_loss/total_val:.4f}",
                    'val_acc': f"{(val_running_corrects.double()/total_val).item()*100:.2f}%"
                })

        val_loss = val_running_loss / total_val if total_val > 0 else 0
        val_acc = (val_running_corrects.double() / total_val).item() if total_val > 0 else 0
        val_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)

        epoch_dur = time.time() - epoch_start
        print(f"Epoch {epoch+1:02d}/{num_epochs:02d} [{epoch_dur:.1f}s] | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% F1: {train_f1:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}% F1: {val_f1:.4f}")

        # Checkpointing based on validation Macro F1 score
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            early_stop_counter = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            early_stop_counter += 1
            if early_stop_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1} (Best Val Macro F1: {best_val_f1:.4f}).")
                break

    time_elapsed = time.time() - start_time
    print(f"Completed in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s. Best validation Macro F1: {best_val_f1:.4f}")

    # 3-Panel Learning Curves Plot: Loss, Accuracy, and Macro F1
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    epochs = range(1, len(history['train_loss']) + 1)

    # 1. Loss Curve
    axes[0].plot(epochs, history['train_loss'], 'o-', label='Train Loss', color='royalblue')
    axes[0].plot(epochs, history['val_loss'], 's-', label='Val Loss', color='crimson')
    axes[0].set_title(f"{model_name}: Loss vs Epochs")
    axes[0].set_xlabel("Epochs")
    axes[0].set_ylabel("Cross Entropy Loss")
    axes[0].legend()

    # 2. Accuracy Curve
    axes[1].plot(epochs, [a * 100 for a in history['train_acc']], 'o-', label='Train Acc', color='royalblue')
    axes[1].plot(epochs, [a * 100 for a in history['val_acc']], 's-', label='Val Acc', color='crimson')
    axes[1].set_title(f"{model_name}: Accuracy vs Epochs")
    axes[1].set_xlabel("Epochs")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()

    # 3. Macro F1 Curve
    axes[2].plot(epochs, history['train_f1'], 'o-', label='Train Macro F1', color='royalblue')
    axes[2].plot(epochs, history['val_f1'], 's-', label='Val Macro F1', color='crimson')
    axes[2].set_title(f"{model_name}: Macro F1 vs Epochs")
    axes[2].set_xlabel("Epochs")
    axes[2].set_ylabel("Macro F1-Score")
    axes[2].set_ylim([0, 1.05])
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{slug}_learning_curves.png", dpi=300)
    plt.show()

    if checkpoint_path.exists():
        model.load_state_dict(torch.load(checkpoint_path))
    return model, history


In [ ]:
def evaluate_and_record(y_true, y_pred, y_prob, model_name, class_names, results_dict):
    """Compute classification metrics, plot confusion matrix and ROC curves, and store results."""
    slug = model_name.lower().replace(" ", "_").replace("-", "_")

    print(f"\n=======================================================")
    print(f"           {model_name.upper()} EVALUATION")
    print(f"=======================================================")
    rep_text = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    print(rep_text)

    with open(REPORTS_DIR / f"{slug}_classification_report.txt", "w") as f:
        f.write(rep_text)

    rep_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    pd.DataFrame(rep_dict).transpose().to_csv(REPORTS_DIR / f"{slug}_classification_report.csv")

    acc = np.mean(y_true == y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    weighted_f1 = f1_score(y_true, y_pred, average='weighted')

    np.savez_compressed(
        REPORTS_DIR / f"{slug}_predictions.npz",
        y_true=y_true,
        y_pred=y_pred,
        y_prob=y_prob
    )

    results_dict[model_name] = {
        'Accuracy': acc,
        'Macro F1': macro_f1,
        'Weighted F1': weighted_f1,
        'y_true': y_true,
        'y_pred': y_pred
    }

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 1. Normalized Confusion Matrix
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0], cbar=False)
    axes[0].set_title(f"{model_name} - Confusion Matrix")
    axes[0].set_xlabel("Predicted Label")
    axes[0].set_ylabel("True Label")
    axes[0].tick_params(axis='x', rotation=45)

    # 2. One-vs-Rest ROC Curve
    n_classes = len(class_names)
    y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))
    fpr, tpr, roc_auc = dict(), dict(), dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    fpr["macro"], tpr["macro"], _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    axes[1].plot(fpr["macro"], tpr["macro"], label=f'Macro-average (AUC = {roc_auc["macro"]:.3f})', color='deeppink', linestyle=':', linewidth=2.5)
    colors = plt.cm.tab20(np.linspace(0, 1, n_classes))
    for i, color in zip(range(n_classes), colors):
        axes[1].plot(fpr[i], tpr[i], color=color, lw=1.2, label=f'{class_names[i]} ({roc_auc[i]:.2f})')
    axes[1].plot([0, 1], [0, 1], 'k--', lw=1.2)
    axes[1].set_title(f"{model_name} - One-vs-Rest ROC Curves")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{slug}_confusion_matrix_and_roc.png", dpi=300)
    plt.show()


In [ ]:
def evaluate_pytorch_model(model, test_loader):
    """Run inference across test dataset and collect predictions and softmax probabilities."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    use_amp = (device.type == 'cuda')

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Inference", leave=False):
            inputs = inputs.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                outputs = model(inputs)
                probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.concatenate(all_labels), np.concatenate(all_preds), np.concatenate(all_probs)

def cleanup_gpu_memory(model_var_names):
    """Reclaim PyTorch and CUDA memory allocations."""
    for name in model_var_names:
        if name in globals():
            del globals()[name]
    torch.cuda.empty_cache()
    gc.collect()


---
## Model 1: Custom Convolutional Neural Network
A baseline 4-stage convolutional neural network with batch normalization, LeakyReLU non-linearities, spatial max-pooling, and dropout regularization.


In [ ]:
class SmallCustomCNN(nn.Module):
    def __init__(self, num_classes=13, dropout_rate=0.3):
        super(SmallCustomCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.global_pool(self.features(x)))

# Hyperparameter candidate screening
hparam_candidates = [
    {"lr": 1e-3, "optimizer": "adamw", "weight_decay": 1e-4, "dropout": 0.3},
    {"lr": 5e-4, "optimizer": "adamw", "weight_decay": 1e-3, "dropout": 0.2}
]

best_lr, best_wd, best_drop = 1e-3, 1e-4, 0.3
best_val_score = 0.0

for cand in hparam_candidates:
    trial_model = SmallCustomCNN(num_classes=NUM_CLASSES, dropout_rate=cand['dropout']).to(device)
    opt = optim.AdamW(trial_model.parameters(), lr=cand['lr'], weight_decay=cand['weight_decay'])
    crit = nn.CrossEntropyLoss()
    use_amp = (device.type == 'cuda')
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    trial_model.train()
    for b_idx, (inputs, labels) in enumerate(train_loader):
        if b_idx >= 30:
            break
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            loss = crit(trial_model(inputs), labels)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

    trial_model.eval()
    corr, tot = 0, 0
    with torch.no_grad():
        for v_idx, (inputs, labels) in enumerate(val_loader):
            if v_idx >= 15:
                break
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                preds = torch.max(trial_model(inputs), 1)[1]
            corr += (preds == labels).sum().item()
            tot += labels.size(0)
    v_acc = corr / tot if tot > 0 else 0
    if v_acc > best_val_score:
        best_val_score = v_acc
        best_lr, best_wd, best_drop = cand['lr'], cand['weight_decay'], cand['dropout']

del trial_model, opt, scaler
torch.cuda.empty_cache()


In [ ]:
# Model 1 Training and Evaluation
custom_cnn = SmallCustomCNN(num_classes=NUM_CLASSES, dropout_rate=best_drop).to(device)
criterion_cnn = nn.CrossEntropyLoss()
optimizer_cnn = optim.AdamW(custom_cnn.parameters(), lr=best_lr, weight_decay=best_wd)
scheduler_cnn = optim.lr_scheduler.CosineAnnealingLR(optimizer_cnn, T_max=15)

custom_cnn, history_cnn = train_pytorch_pipeline(
    custom_cnn, train_loader, val_loader, criterion_cnn, optimizer_cnn,
    scheduler=scheduler_cnn, num_epochs=15, patience=4, model_name="Small Custom CNN"
)

y_true_cnn, y_pred_cnn, y_prob_cnn = evaluate_pytorch_model(custom_cnn, test_loader)
evaluate_and_record(y_true_cnn, y_pred_cnn, y_prob_cnn, "Small Custom CNN", class_names, all_model_results)

cleanup_gpu_memory(['custom_cnn', 'optimizer_cnn', 'scheduler_cnn', 'criterion_cnn'])


---
## Model 2: MobileNetV3-Large
Pretrained lightweight architecture using depthwise separable convolutions, hard-swish non-linearities, and squeeze-and-excitation attention blocks.


In [ ]:
def build_mobilenet_v3(num_classes=13):
    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

mobilenet_model = build_mobilenet_v3(num_classes=NUM_CLASSES).to(device)
criterion_mb = nn.CrossEntropyLoss()
optimizer_mb = optim.AdamW(mobilenet_model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler_mb = optim.lr_scheduler.CosineAnnealingLR(optimizer_mb, T_max=15)

mobilenet_model, history_mb = train_pytorch_pipeline(
    mobilenet_model, train_loader, val_loader, criterion_mb, optimizer_mb,
    scheduler=scheduler_mb, num_epochs=15, patience=4, model_name="MobileNetV3-Large"
)

y_true_mb, y_pred_mb, y_prob_mb = evaluate_pytorch_model(mobilenet_model, test_loader)
evaluate_and_record(y_true_mb, y_pred_mb, y_prob_mb, "MobileNetV3-Large", class_names, all_model_results)

cleanup_gpu_memory(['mobilenet_model', 'optimizer_mb', 'scheduler_mb', 'criterion_mb'])


---
## Model 3: YOLOv8 Classifier
Ultralytics YOLOv8 nano classification architecture (`yolov8n-cls`), utilizing modified CSPDarkNet backbone blocks.


In [ ]:
yolo8_model = YOLO("yolov8n-cls.pt")

yolo8_results = yolo8_model.train(
    data=str(Path("data").resolve()),
    epochs=15,
    imgsz=224,
    batch=BATCH_SIZE,
    device=0 if torch.cuda.is_available() else 'cpu',
    seed=SEED,
    workers=2,
    project="runs_benchmark",
    name="yolov8n_cls",
    exist_ok=True
)

yolo8_best_weights = Path("runs_benchmark/yolov8n_cls/weights/best.pt")
if yolo8_best_weights.exists():
    shutil.copy2(yolo8_best_weights, MODELS_DIR / "best_yolo8n_cls.pt")

def evaluate_yolo_predictions(model, test_dir, class_names):
    test_path = Path(test_dir)
    labels, preds, probs = [], [], []
    c2i = {name: i for i, name in enumerate(class_names)}

    for cname in tqdm(class_names, desc="YOLO Inference", leave=False):
        cfolder = test_path / cname
        if not cfolder.exists():
            continue
        imgs = [f for f in cfolder.iterdir() if f.suffix.lower() in VALID_EXTENSIONS]
        for img_p in imgs:
            res = model(str(img_p), verbose=False)[0]
            p_arr = res.probs.data.cpu().numpy()
            labels.append(c2i[cname])
            preds.append(int(np.argmax(p_arr)))
            probs.append(p_arr)

    return np.array(labels), np.array(preds), np.array(probs)

y_true_yolo8, y_pred_yolo8, y_prob_yolo8 = evaluate_yolo_predictions(yolo8_model, "data/test", class_names)
evaluate_and_record(y_true_yolo8, y_pred_yolo8, y_prob_yolo8, "YOLO8n-cls", class_names, all_model_results)

cleanup_gpu_memory(['yolo8_model', 'yolo8_results'])


---
## Model 4: YOLO26 Classifier
Ultralytics official YOLO26 classification architecture (`yolo26n-cls.pt`).


In [ ]:
yolo26_model = YOLO("yolo26n-cls.pt")

yolo26_results = yolo26_model.train(
    data=str(Path("data").resolve()),
    epochs=15,
    imgsz=224,
    batch=BATCH_SIZE,
    device=0 if torch.cuda.is_available() else 'cpu',
    seed=SEED,
    workers=2,
    project="runs_benchmark",
    name="yolo26n_cls",
    exist_ok=True
)

yolo26_best_weights = Path("runs_benchmark/yolo26n_cls/weights/best.pt")
if yolo26_best_weights.exists():
    shutil.copy2(yolo26_best_weights, MODELS_DIR / "best_yolo26n_cls.pt")

y_true_yolo26, y_pred_yolo26, y_prob_yolo26 = evaluate_yolo_predictions(yolo26_model, "data/test", class_names)
evaluate_and_record(y_true_yolo26, y_pred_yolo26, y_prob_yolo26, "YOLO26n-cls", class_names, all_model_results)

cleanup_gpu_memory(['yolo26_model', 'yolo26_results'])


---
## Model 5: EfficientNet-B0
Architecture utilizing compound scaling to jointly optimize network depth, width, and input resolution.


In [ ]:
def build_efficientnet_b0(num_classes=13):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

effnet_model = build_efficientnet_b0(num_classes=NUM_CLASSES).to(device)
criterion_eff = nn.CrossEntropyLoss()
optimizer_eff = optim.AdamW(effnet_model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler_eff = optim.lr_scheduler.CosineAnnealingLR(optimizer_eff, T_max=15)

effnet_model, history_eff = train_pytorch_pipeline(
    effnet_model, train_loader, val_loader, criterion_eff, optimizer_eff,
    scheduler=scheduler_eff, num_epochs=15, patience=4, model_name="EfficientNet-B0"
)

y_true_eff, y_pred_eff, y_prob_eff = evaluate_pytorch_model(effnet_model, test_loader)
evaluate_and_record(y_true_eff, y_pred_eff, y_prob_eff, "EfficientNet-B0", class_names, all_model_results)

cleanup_gpu_memory(['effnet_model', 'optimizer_eff', 'scheduler_eff', 'criterion_eff'])


---
## Model 6: ResNet-50
Deep residual network utilizing 50 layers with residual shortcut connections to mitigate vanishing gradients in deeper network topologies.


In [ ]:
def build_resnet50(num_classes=13):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

resnet_model = build_resnet50(num_classes=NUM_CLASSES).to(device)
criterion_res = nn.CrossEntropyLoss()
optimizer_res = optim.AdamW(resnet_model.parameters(), lr=1.5e-4, weight_decay=1e-4)
scheduler_res = optim.lr_scheduler.CosineAnnealingLR(optimizer_res, T_max=15)

resnet_model, history_res = train_pytorch_pipeline(
    resnet_model, train_loader, val_loader, criterion_res, optimizer_res,
    scheduler=scheduler_res, num_epochs=15, patience=4, model_name="ResNet50"
)

y_true_res, y_pred_res, y_prob_res = evaluate_pytorch_model(resnet_model, test_loader)
evaluate_and_record(y_true_res, y_pred_res, y_prob_res, "ResNet50", class_names, all_model_results)

cleanup_gpu_memory(['resnet_model', 'optimizer_res', 'scheduler_res', 'criterion_res'])


---
## 7. Comparative Benchmark and Results Summary
Consolidated comparative analysis across all six evaluated architectures on the unseen test set.


In [ ]:
# Benchmark table compilation
summary_rows = []
for model_name, metrics in all_model_results.items():
    summary_rows.append({
        "Model Architecture": model_name,
        "Accuracy (%)": f"{metrics['Accuracy'] * 100:.2f}%",
        "Macro F1-Score": f"{metrics['Macro F1']:.4f}",
        "Weighted F1-Score": f"{metrics['Weighted F1']:.4f}"
    })

df_benchmark = pd.DataFrame(summary_rows)
display(df_benchmark)

benchmark_csv_path = OUTPUT_DIR / "final_benchmark_summary.csv"
df_benchmark.to_csv(benchmark_csv_path, index=False)


In [ ]:
# Comparative visualization
if not df_benchmark.empty:
    chart_data = pd.DataFrame([
        {
            "Model": name,
            "Accuracy (%)": metrics['Accuracy'] * 100,
            "Macro F1": metrics['Macro F1']
        }
        for name, metrics in all_model_results.items()
    ])

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    sns.barplot(data=chart_data, x="Model", y="Accuracy (%)", palette="Blues_d", ax=axes[0])
    axes[0].set_title("Test Accuracy Across Architectures", fontsize=13)
    axes[0].set_ylim([0, 100])
    axes[0].tick_params(axis='x', rotation=30)
    for p in axes[0].patches:
        axes[0].annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() - 7),
                         ha='center', va='center', color='white', fontweight='bold')

    sns.barplot(data=chart_data, x="Model", y="Macro F1", palette="Greens_d", ax=axes[1])
    axes[1].set_title("Macro F1-Score Across Architectures", fontsize=13)
    axes[1].set_ylim([0, 1.0])
    axes[1].tick_params(axis='x', rotation=30)
    for p in axes[1].patches:
        axes[1].annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2., p.get_height() - 0.07),
                         ha='center', va='center', color='white', fontweight='bold')

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "final_benchmark_comparison.png", dpi=300)
    plt.show()


## 8. Export Outputs


In [ ]:
# Compress outputs directory
zip_filename = "fruit_classification_outputs.zip"
shutil.make_archive("fruit_classification_outputs", 'zip', OUTPUT_DIR)
print(f"Outputs compressed to: {zip_filename}")

if 'google.colab' in sys.modules:
    try:
        from google.colab import files
        files.download(zip_filename)
    except Exception as e:
        print(f"Colab download skipped/error: {e}")
